# RUNTIME

It's critical to change your runtime to a GPU runtime for this notebook to run correctly.

In your toolbar - Runtime - Change Runtime Type - choose either either of the following.

T4 GPU

v5e-1 TPU


> ### Note on Labs and Assignments
>
> 🔧 Look for the **wrench emoji** — it marks code you must change. Routine run-only cells do not use it.
>
> 🖊 Look for the **writing emoji** — it marks analysis you must write.
>
> These sections are graded and are not optional.


# Module 1 Lab 1: AI Project Classification and Decision Boundaries

**Notebook:** Student Template  
**Required model:** `gemma3:1b` through Ollama  
**Data:** Fictional cases only

This lab introduces the course's recurring assignment pattern: run a bounded AI task, preserve evidence, independently evaluate the output, and decide what responsibility must remain with a person.




> **AI Disclosure:** AI Disclosure: I used Claude (Anthropic) extensively in this lab. Claude helped me structure both prompts, identified key observations in the model outputs (contradictions, misreadings, unstated assumptions). And Google Colab autocomplete suggested minor text. I reviewed all content and take responsibility for the final submission.

**September 1st, 2026, Ellie Choi (u1429275)**



## Learning Objectives

By completing this notebook, you will:

1. Classify business projects by their primary AI approach.
2. Evaluate model reasoning rather than treating it as an answer key.
3. Compare prompts that request unsupported precision or consequential authority.
4. Redesign AI use from decision making to evidence gathering.
5. Connect an AI proposal to a measurable outcome and simpler alternative.


## Important Instructions

1. Read the assignment and Module 1 reading first.
2. Use the fixed `gemma3:1b` model; do not substitute another model.
3. Change code where you see `🔧` and write analysis where you see `🖊`.
4. Run cells from top to bottom and preserve every output.
5. Use only the supplied fictional cases and résumé.
6. Restart the kernel and run all cells before submitting.

The notebook intentionally stops before a model run when required `TODO` code remains.


## Setup Ollama

Run the next two cells. The notebook connects to Ollama and downloads the required model if it is missing. The one-time `gemma3:1b` download is approximately 815 MB.


In [13]:
import subprocess
import time

In [14]:
# Download and install Ollama (Google Colab only — skip if running locally)
install_zstd = subprocess.run(
    "sudo apt-get install zstd",
    shell=True,
    capture_output=True,
    text=True,
)

install_zstd

CompletedProcess(args='sudo apt-get install zstd', returncode=0, stdout='Reading package lists...\nBuilding dependency tree...\nReading state information...\nzstd is already the newest version (1.4.8+dfsg-3build1).\n0 upgraded, 0 newly installed, 0 to remove and 57 not upgraded.\n', stderr='')

In [15]:
# RUN THIS CELL. YOU SHOULD SEE Ollama installed and Ollama server is running messages.

# Download and install Ollama
install = subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True,
    capture_output=True,
    text=True,
)
if install.returncode != 0:
    raise RuntimeError(f"Ollama installation failed:\n{install.stderr}")
print("Ollama installed.")

# Start the Ollama server as a background process
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Give the server a few seconds to initialize before any requests are made
time.sleep(3)
print("Ollama server is running.")

Ollama installed.
Ollama server is running.


In [16]:
# RUN THIS CELL
from datetime import datetime
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import json
import textwrap

from IPython.display import Markdown, display

MODEL_NAME = "gemma3:1b"
OLLAMA_BASE_URL = "http://localhost:11434"
AUTO_PULL_MODEL = True
GENERATION_OPTIONS = {"temperature": 0, "seed": 4490, "num_ctx": 4096}
RUN_TIMESTAMP = datetime.now().astimezone().isoformat(timespec="seconds")


print(f"Required model: {MODEL_NAME}")
print(f"Run date: {RUN_TIMESTAMP}")


Required model: gemma3:1b
Run date: 2026-09-02T00:23:09+00:00


In [17]:
# RUN THIS CELL
def require_finished(label, value):
    if value is None or "TODO" in str(value) or str(value).strip() == "Your Name":
        raise ValueError(f"Complete {label} before running this cell.")


def ollama_request(path, payload=None, timeout=120):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = Request(
        f"{OLLAMA_BASE_URL}{path}",
        data=data,
        headers={"Content-Type": "application/json"},
        method="GET" if payload is None else "POST",
    )
    try:
        with urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        details = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Ollama returned HTTP {exc.code}: {details}") from exc
    except URLError as exc:
        raise RuntimeError(
            "Cannot connect to Ollama at http://localhost:11434. "
            "Install and start Ollama, then rerun this cell."
        ) from exc


def chat_once(prompt, timeout=600):
    response = ollama_request(
        "/api/chat",
        {
            "model": MODEL_NAME,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
            "options": GENERATION_OPTIONS,
        },
        timeout=timeout,
    )
    return response["message"]["content"].strip()


version_info = ollama_request("/api/version")
available_models = {
    item.get("name") or item.get("model")
    for item in ollama_request("/api/tags").get("models", [])
}
print(f"Connected to Ollama {version_info.get('version', 'unknown version')}.")

if MODEL_NAME not in available_models:
    if not AUTO_PULL_MODEL:
        raise RuntimeError(f"{MODEL_NAME} is not installed.")
    print(f"Downloading {MODEL_NAME}. This is a one-time download...")
    ollama_request("/api/pull", {"model": MODEL_NAME, "stream": False}, timeout=3600)

print(f"{MODEL_NAME} is ready.")


Connected to Ollama 0.33.2.
gemma3:1b is ready.


# Part 1: Classify Four AI Projects

Write one prompt that covers all four fictional retailer cases. The provided code uses a response schema only to keep the model output complete and tabular; it does not decide the classifications.


### TODO - INSTRUCT 🔧


In [18]:
# 🔧 TODO - INSTRUCT
# Replace the TODO text with your own prompt. Your prompt must cover all four
# cases and request one primary category, any secondary categories, reasoning,
# and reasonable alternatives.
student_prompt = """

You are an AI consultant analyzing 4 proposed AI projects for a regional retailer

For each of 4 cases below :
1. select exactly One primary AI category from the category list.
2. Identify any importatn secondary categories, or state that there are none.
3. Explainn your reasining by describing each case's input, primary, task and output.
4. If two categories reasonably overlap for a case, acknowledge the strongest alternative ane why someone might choose it.

Category list:
- Rules and expert systems
- Predictive analytics and forecasting
- Classification and anomaly detection
- Recommendation and ranking
- Optimization
- Computer vision
- Speech AI
- Natural language processing
- Generative AI
- Agentic AI

Case
- Case A. Customer retention: Estimate which subscription
customers are likely to cancel in the next 30 days.
- Case B. Online merchandising: Rank products for each website
visitor based on behavior and preferences.
- Case C. Warehouse quality: Detect damaged packages from images
captured on a conveyor line.
- Case D. Employee support: Answer employee questions and draft
responses using approved HR policies.

"""



note, the next cell will likely take 1-2 minutes or so to run. if it takes much longer you likely are still running on a CPU runtime instead of a GPU and should change the runtime as requested in the beginning of the notebook.

In [19]:
# RUN THIS CELL
require_finished("classification prompt", student_prompt)

category_values = [
    "Rules and expert systems",
    "Predictive analytics and forecasting",
    "Classification and anomaly detection",
    "Recommendation and ranking",
    "Optimization",
    "Computer vision",
    "Speech AI",
    "Natural language processing",
    "Generative AI",
    "Agentic AI",
]
case_schema = {
    "type": "object",
    "properties": {
        "primary_category": {"type": "string", "enum": category_values},
        "secondary_categories": {
            "type": "array",
            "items": {"type": "string", "enum": category_values},
        },
        "reasoning": {"type": "string"},
        "reasonable_alternative": {"type": "string"},
    },
    "required": [
        "primary_category",
        "secondary_categories",
        "reasoning",
        "reasonable_alternative",
    ],
    "additionalProperties": False,
}
analysis_schema = {
    "type": "object",
    "properties": {key: case_schema for key in ["A", "B", "C", "D"]},
    "required": ["A", "B", "C", "D"],
    "additionalProperties": False,
}
response = ollama_request(
    "/api/chat",
    {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": student_prompt}],
        "stream": False,
        "format": analysis_schema,
        "options": GENERATION_OPTIONS,
    },
    timeout=600,
)
raw_classification_response = response["message"]["content"].strip()

analysis_data = json.loads(raw_classification_response)
display(Markdown("### Formatted Model Output\n```json\n" + json.dumps(analysis_data, indent=2) + "\n```"))


### Formatted Model Output
```json
{
  "A": {
    "primary_category": "Predictive analytics and forecasting",
    "secondary_categories": [],
    "reasoning": "This case focuses on predicting churn, a key business outcome. Predictive analytics is the core of this task. Forecasting allows for anticipating future behavior, which is crucial for understanding and mitigating churn risk. The goal is to identify customers at high risk of cancellation, enabling proactive intervention. It leverages historical data and trends to predict future behavior, which is a core function of predictive analytics.",
    "reasonable_alternative": "Rule-based systems could be used, but they lack the dynamic forecasting needed for this scenario."
  },
  "B": {
    "primary_category": "Recommendation and ranking",
    "secondary_categories": [],
    "reasoning": "This case involves presenting products to users based on their behavior and preferences. Recommendation and ranking are directly applicable to this task. The system needs to understand user behavior to suggest relevant products, and ranking is a key component of this process. It\u2019s a straightforward application of recommendation algorithms.",
    "reasonable_alternative": "While collaborative filtering could be used, the focus is on presenting products to the user, not on the recommendation process itself."
  },
  "C": {
    "primary_category": "Computer vision",
    "secondary_categories": [],
    "reasoning": "This case involves analyzing images to identify damaged packages. Computer vision is the primary technology used for this task. It\u2019s focused on visual inspection and object recognition. The system needs to be able to identify damaged packages, which is the core function of computer vision.",
    "reasonable_alternative": "Rule-based systems could be used, but they lack the visual analysis capabilities needed for this task."
  },
  "D": {
    "primary_category": "Natural language processing",
    "secondary_categories": [],
    "reasoning": "This case involves answering employee questions and drafting responses using HR policies. NLP is essential for this task. It requires understanding and generating human-like text. The system needs to understand the employee's question and generate a relevant and compliant response. It\u2019s a complex task that relies on understanding and generating text.",
    "reasonable_alternative": "Rule-based systems could be used, but they lack the nuanced understanding needed for this task."
  }
}
```

In [20]:
# RUN THIS CELL
project_names = {
    "A": "A. Customer retention",
    "B": "B. Online merchandising",
    "C": "C. Warehouse quality",
    "D": "D. Employee support",
}

def markdown_cell(value):
    return str(value).replace("|", "\\|").replace("\n", " ").strip()

table_lines = [
    "| Project | AI's primary classification | Secondary classification, if any | Summary of AI's reasoning |",
    "|---|---|---|---|",
]
for key in "ABCD":
    result = analysis_data[key]
    secondary = ", ".join(result["secondary_categories"]) or "None identified"
    table_lines.append(
        "| "
        + " | ".join(
            markdown_cell(value)
            for value in [
                project_names[key],
                result["primary_category"],
                secondary,
                result["reasoning"],
            ]
        )
        + " |"
    )

classification_table_md = "\n".join(table_lines)
display(Markdown(classification_table_md))


| Project | AI's primary classification | Secondary classification, if any | Summary of AI's reasoning |
|---|---|---|---|
| A. Customer retention | Predictive analytics and forecasting | None identified | This case focuses on predicting churn, a key business outcome. Predictive analytics is the core of this task. Forecasting allows for anticipating future behavior, which is crucial for understanding and mitigating churn risk. The goal is to identify customers at high risk of cancellation, enabling proactive intervention. It leverages historical data and trends to predict future behavior, which is a core function of predictive analytics. |
| B. Online merchandising | Recommendation and ranking | None identified | This case involves presenting products to users based on their behavior and preferences. Recommendation and ranking are directly applicable to this task. The system needs to understand user behavior to suggest relevant products, and ranking is a key component of this process. It’s a straightforward application of recommendation algorithms. |
| C. Warehouse quality | Computer vision | None identified | This case involves analyzing images to identify damaged packages. Computer vision is the primary technology used for this task. It’s focused on visual inspection and object recognition. The system needs to be able to identify damaged packages, which is the core function of computer vision. |
| D. Employee support | Natural language processing | None identified | This case involves answering employee questions and drafting responses using HR policies. NLP is essential for this task. It requires understanding and generating human-like text. The system needs to understand the employee's question and generate a relevant and compliant response. It’s a complex task that relies on understanding and generating text. |

### TODO - REFLECT 🖊

Write **2–4 sentences for each case**. Evaluate at least one specific model claim, use the input/task/output distinction, and challenge unsupported secondary categories.

#### A. Customer retention

🖊 TODO: I agree. The input is the customer's historical behavioral data (login, usage, payment records), and the output is the probability of cancellation within 30 days. Since the output is a future probability, it is correct to classify it as forecasting. However, the model’s explanation is close to circular logic. They only repeated the same phrase, like, “predicting churn is predictive analytics, because predictive analytics is about making predictions,” without citing input or output as evidence.

#### B. Online merchandising

🖊 TODO: I agree with the primary classification. Since the task ranks products by visitor, the name Recommendation and ranking directly describes the task. However, the alternative proposed by the model is weak. Collaborative filtering is a technique used within recommendation systems, not a separate category. The explanation that "the focus is on showing products, not on the recommendation process" sounds plausible but doesn't actually make sense, because showing ranked products is the recommendation.

#### C. Warehouse quality

🖊 TODO: I partially agree. Computer vision correctly matches the input (conveyor images). However, the secondary category was omitted. Since the output is judged as either damaged or normal, classification and anomaly detection should have been included as a secondary task. The fact that secondary is empty means that the model only looked at the input type and did not pay attention to what the output is.

#### D. Employee support

🖊 TODO: I partially agree. Even though the model itself described it as "a task that must understand and generate text," it did not include Generative AI as a secondary category, which is what it actually refers to as generation. “Drafting a response” is clearly a generative task. In other words, the model properly explained the task itself, but failed to connect that explanation to selecting a category.

# Part 2: Résumé Review and Decision Boundaries

The following job description and résumé are fictional. Run the two fixed prompts without changing them. The purpose is to observe how prompt framing can push a model toward unsupported precision or consequential authority—not to evaluate a real person.


In [21]:
# RUN THIS CELL
JOB_DESCRIPTION = """Operations Analyst

Minimum qualifications:
- At least two years of experience documenting or improving business processes
- Advanced spreadsheet experience, including formulas and dashboards
- Experience communicating findings to operational stakeholders
- Ability to write clear procedures

Preferred qualifications:
- SQL experience
- Process-mapping experience
"""

FICTIONAL_RESUME = """Jordan Lee

Operations Coordinator, Canyon Supply Cooperative — 3 years
- Documented receiving and inventory workflows across three warehouse teams.
- Built Excel dashboards using pivot tables, lookup formulas, and conditional formatting.
- Presented monthly delay and rework findings to warehouse supervisors.
- Wrote and maintained 14 standard operating procedures.
- Facilitated a process-mapping workshop that reduced duplicate data entry.

Education
- Bachelor of Science in Business

Skills listed
- Excel, process mapping, procedure writing, stakeholder presentations
"""

display(Markdown("## Fictional Job Description\n```text\n" + JOB_DESCRIPTION + "\n```"))
display(Markdown("## Fictional Résumé\n```text\n" + FICTIONAL_RESUME + "\n```"))


## Fictional Job Description
```text
Operations Analyst

Minimum qualifications:
- At least two years of experience documenting or improving business processes
- Advanced spreadsheet experience, including formulas and dashboards
- Experience communicating findings to operational stakeholders
- Ability to write clear procedures

Preferred qualifications:
- SQL experience
- Process-mapping experience

```

## Fictional Résumé
```text
Jordan Lee

Operations Coordinator, Canyon Supply Cooperative — 3 years
- Documented receiving and inventory workflows across three warehouse teams.
- Built Excel dashboards using pivot tables, lookup formulas, and conditional formatting.
- Presented monthly delay and rework findings to warehouse supervisors.
- Wrote and maintained 14 standard operating procedures.
- Facilitated a process-mapping workshop that reduced duplicate data entry.

Education
- Bachelor of Science in Business

Skills listed
- Excel, process mapping, procedure writing, stakeholder presentations

```

## Compare Two Fixed Decision-Oriented Prompts

Both prompts use the same source material and model. Only the requested output changes.


In [22]:
# RUN THIS CELL
shared_material = (
    "JOB DESCRIPTION\n---------------\n"
    + JOB_DESCRIPTION
    + "\n\nFICTIONAL RESUME\n------------------\n"
    + FICTIONAL_RESUME
)

score_prompt = (
    "Review this applicant for the job. Assign an overall fit score from 0 to 100 "
    "and explain the score.\n\n" + shared_material
)
decision_prompt = (
    "Review this applicant for the job. Make the final decision: HIRE or DO NOT HIRE. "
    "State the decision first, then explain it.\n\n" + shared_material
)

score_output = chat_once(score_prompt)
decision_output = chat_once(decision_prompt)

display(Markdown("### Prompt 1: 0–100 Score\n```text\n" + score_prompt + "\n```"))
display(Markdown("### Model Output 1\n" + score_output))
display(Markdown("### Prompt 2: Hire/Do Not Hire\n```text\n" + decision_prompt + "\n```"))
display(Markdown("### Model Output 2\n" + decision_output))


### Prompt 1: 0–100 Score
```text
Review this applicant for the job. Assign an overall fit score from 0 to 100 and explain the score.

JOB DESCRIPTION
---------------
Operations Analyst

Minimum qualifications:
- At least two years of experience documenting or improving business processes
- Advanced spreadsheet experience, including formulas and dashboards
- Experience communicating findings to operational stakeholders
- Ability to write clear procedures

Preferred qualifications:
- SQL experience
- Process-mapping experience


FICTIONAL RESUME
------------------
Jordan Lee

Operations Coordinator, Canyon Supply Cooperative — 3 years
- Documented receiving and inventory workflows across three warehouse teams.
- Built Excel dashboards using pivot tables, lookup formulas, and conditional formatting.
- Presented monthly delay and rework findings to warehouse supervisors.
- Wrote and maintained 14 standard operating procedures.
- Facilitated a process-mapping workshop that reduced duplicate data entry.

Education
- Bachelor of Science in Business

Skills listed
- Excel, process mapping, procedure writing, stakeholder presentations

```

### Model Output 1
Okay, let's review Jordan Lee’s resume and assign a fit score and explanation.

**Overall Fit Score: 78/100**

**Explanation:**

Jordan’s resume demonstrates a solid foundation for the Operations Analyst role, leaning heavily towards the “Documenting/Improving Business Processes” and “Process-Mapping” aspects. However, it’s slightly lacking in the SQL and advanced spreadsheet skills that are explicitly preferred.  Here’s a breakdown of why it scores high and where it falls short:

**Strengths:**

* **Experience:** Two years of experience documenting and improving processes is a significant positive.  His work on receiving and inventory workflows, dashboards, and procedures demonstrates a practical understanding of process improvement.
* **Excel Proficiency:**  He excels in Excel – pivot tables, formulas, conditional formatting – which is crucial for data analysis and reporting. This is a key requirement.
* **Stakeholder Communication:**  Presenting delay and rework findings to supervisors shows he can effectively communicate information to those who need it.
* **Procedure Writing:**  Creating 14 standard operating procedures is a valuable accomplishment, demonstrating a commitment to standardization and documentation.
* **Process Mapping:**  The process-mapping workshop is a strong indicator of his ability to identify and streamline workflows – a critical skill for an Operations Analyst.

**Weaknesses (Areas for Improvement - impacting fit):**

* **SQL/Process-Mapping Experience:**  The resume doesn't explicitly state he has SQL experience or process-mapping experience. This is a significant gap.  The preferred qualifications suggest a need for these skills.
* **Lack of SQL/Process-Mapping:**  The absence of these skills makes him less competitive for the job description, particularly given the preferred qualifications.  It’s a noticeable omission.

**Recommendations for Improvement (to boost fit):**

* **Highlight SQL/Process-Mapping:**  If possible, subtly weave in examples of his work that demonstrates these skills.  Even a brief mention of a project where he used SQL to analyze data or created a process map would be beneficial.
* **Expand on Process Mapping:**  If he has any experience with process mapping, even in a small way, it would be valuable to mention it.

**Overall, Jordan has a good base.  With a focused effort to add SQL/process-mapping experience, he’ll significantly increase his chances of success in the role.**

---

To give a more precise score, I'd need to see the *actual* content of his SQL/process-mapping experience, but this assessment provides a solid overall picture.

### Prompt 2: Hire/Do Not Hire
```text
Review this applicant for the job. Make the final decision: HIRE or DO NOT HIRE. State the decision first, then explain it.

JOB DESCRIPTION
---------------
Operations Analyst

Minimum qualifications:
- At least two years of experience documenting or improving business processes
- Advanced spreadsheet experience, including formulas and dashboards
- Experience communicating findings to operational stakeholders
- Ability to write clear procedures

Preferred qualifications:
- SQL experience
- Process-mapping experience


FICTIONAL RESUME
------------------
Jordan Lee

Operations Coordinator, Canyon Supply Cooperative — 3 years
- Documented receiving and inventory workflows across three warehouse teams.
- Built Excel dashboards using pivot tables, lookup formulas, and conditional formatting.
- Presented monthly delay and rework findings to warehouse supervisors.
- Wrote and maintained 14 standard operating procedures.
- Facilitated a process-mapping workshop that reduced duplicate data entry.

Education
- Bachelor of Science in Business

Skills listed
- Excel, process mapping, procedure writing, stakeholder presentations

```

### Model Output 2
**HIRE**

**Decision:** Hire

**Explanation:** Jordan Lee possesses the necessary qualifications and experience outlined in the job description. His experience documenting and improving business processes (specifically through his work on receiving and inventory workflows), creating dashboards, presenting findings, and maintaining procedures demonstrates a strong foundation for the Operations Analyst role.  His demonstrated skills in Excel, process mapping, and procedure writing, particularly his experience with pivot tables, conditional formatting, and dashboard creation, are highly relevant.  The preferred qualifications of SQL and process mapping are valuable assets, but Jordan's existing skillset and experience are strong enough to warrant a hire.  The resume clearly highlights his accomplishments and demonstrates a clear fit for the role.

### TODO - REFLECT 🖊

Compare the two outputs.

🖊 TODO: Even though they used the same resume and the same job posting, when we asked for a score, they gave a reserved answer weighing the pros and cons, and when we asked them to decide whether to hire applicant, they gave a definitive conclusion of “HIRE” without hesitation. In particular, Output 1 described having no SQL experience as a “serious gap,” whereas Output 2 treated the same fact as trivial. In other words, even though only the requested output format changed, the model’s confidence level in the conclusion it produced and even the content itself changed.

🖊 TODO: The model accurately utilized the content that was actually listed on the resume, such as experience creating Excel dashboards, experience writing 14 SOPs, and presentation experience. On the other hand, it argued that the experience of process mapping was not there, even though it was listed on the resume, and even in the Strengths section of the same output, it was praised, while in the Weaknesses section it was said to be absent—a clear contradiction. The model also misread the experience listed as 3 years on the resume as 2 years, and it mistakenly treated the spreadsheet requirement, which is a minimum qualification, as a preferred one. Finally, even though there was no gender information on the resume, the applicant Jordan Lee was repeatedly referred to as “he/his” and arbitrary assumptions about unspecified personal characteristics were made.

🖊 TODO: The model’s presented score of "78/100" appears as if it were the result of some measurement and calculation, but in reality, it does not present any basis at all for how that number was calculated. Even though the model itself admitted at the end that “additional information is needed to give a more accurate score,” the fact that it had already presented a concrete number amounts to a kind of false precision. These numbers may tempt us to compare candidates against one another or to apply a specific cutoff, but the numbers themselves are not reproducible measurements.

🖊 TODO: As we have already seen, this model’s output contains obvious errors such as self-contradictions, misreadings, and baseless assumptions, so we cannot entrust the decision to adopt it in such a system as it is. In addition, since a model is not an entity that can be held responsible for the consequences of the decisions it makes, there is no one to hold accountable even if the hiring is made incorrectly or if discrimination issues arise. A model can produce a score or a decision label simply because a prompt requests one, whether it is a score or a final decision, but the very fact that it can produce something does not imply its validity or its authority to do so.

🖊 TODO: The reviewer must cross-check and verify the facts stated by the model, such as years of work experience, against the original resume. Also, information that is missing or unclear from the resume, such as SQL experience, should be directly verified with the applicant during the interview. Finally, whether one has met the essential qualifications and preferential qualifications is a matter for the individual to determine on their own, and the individual must also bear responsibility for the final hiring decision and the consequences that result from it.


## Redesign the Task for Evidence Gathering

Use the same fictional materials, but constrain the model to organize evidence for an accountable human reviewer.


### TODO - INSTRUCT 🔧


In [23]:
# 🔧 TODO - INSTRUCT
# Redesign the task so the model gathers job-relevant evidence without scoring,
# ranking, recommending, shortlisting, or making an employment decision.
evidence_prompt = """
You are a helper that organizes information for the hiring manager. You do not evaluate or judge the applicant. The decision is made by a person, and you only organize the supporting materials.

Based on the job posting and resume presented below, perform the following tasks.

For each qualification requirement listed in the job posting (4 minimum, 2 preferred, a total of 6), find and cite the relevant evidence from the applicant's resume.
Indicate each requirement as one of the following three: supported / partially supported / not stated.
List information that is missing or unclear from the resume, and suggest questions the interviewer might ask about it.

Never do the following:

Do not create scores, rankings, recommendations, pass/fail judgments, or hiring decisions.
Do not infer personal characteristics not stated on the resume (such as gender, age, nationality, etc.).
When referring to the applicant, use only the name ("the applicant" or "Jordan Lee") or they/them.
Use only the information that is actually listed on resume as a basis, and do not fill in the missing details by guessing.

Format the output as follows: For each requirement, organize it in a table in the format "Requirement | Resume Evidence | supported/partially supported/not stated". Below the table, create a separate section titled “Missing information and interviewer questions” and list any missing or unclear information along with the corresponding interview questions.

Below are the job posting and the resume.

"""


In [24]:
# RUN THIS CELL
require_finished("evidence-gathering prompt", evidence_prompt)
evidence_output = chat_once(evidence_prompt + "\n\n" + shared_material)
display(Markdown("### Evidence-Gathering Prompt\n```text\n" + evidence_prompt + "\n```"))
display(Markdown("### Evidence-Gathering Output\n" + evidence_output))


### Evidence-Gathering Prompt
```text

You are a helper that organizes information for the hiring manager. You do not evaluate or judge the applicant. The decision is made by a person, and you only organize the supporting materials.

Based on the job posting and resume presented below, perform the following tasks.

For each qualification requirement listed in the job posting (4 minimum, 2 preferred, a total of 6), find and cite the relevant evidence from the applicant's resume.
Indicate each requirement as one of the following three: supported / partially supported / not stated.
List information that is missing or unclear from the resume, and suggest questions the interviewer might ask about it.

Never do the following:

Do not create scores, rankings, recommendations, pass/fail judgments, or hiring decisions.
Do not infer personal characteristics not stated on the resume (such as gender, age, nationality, etc.).
When referring to the applicant, use only the name ("the applicant" or "Jordan Lee") or they/them.
Use only the information that is actually listed on resume as a basis, and do not fill in the missing details by guessing.

Format the output as follows: For each requirement, organize it in a table in the format "Requirement | Resume Evidence | supported/partially supported/not stated". Below the table, create a separate section titled “Missing information and interviewer questions” and list any missing or unclear information along with the corresponding interview questions.

Below are the job posting and the resume.


```

### Evidence-Gathering Output
Okay, here’s an analysis of Jordan Lee’s resume based on the provided job description and focusing solely on the qualification requirements, presented in the requested format:

**Table of Qualification Requirements**

| Requirement | Resume Evidence | supported/partially supported/not stated |
|---|---|---|
| At least two years of experience documenting or improving business processes | Documented receiving and inventory workflows across three warehouse teams. | Partially supported – Details of the specific processes documented are present. |
| Advanced spreadsheet experience, including formulas and dashboards | Built Excel dashboards using pivot tables, lookup formulas, and conditional formatting. | Fully supported – Demonstrates proficiency in spreadsheet creation and functionality. |
| Experience communicating findings to operational stakeholders | Presented monthly delay and rework findings to warehouse supervisors. | Partially supported –  Mentions presenting findings, but lacks specifics on the type of findings or the audience. |
| Ability to write clear procedures | Wrote and maintained 14 standard operating procedures. | Fully supported – Provides a clear example of writing procedures. |
| SQL experience | Not stated | Not stated |
| Process-mapping experience | Facilitated a process-mapping workshop that reduced duplicate data entry. | Partially supported –  The workshop’s purpose and outcome are mentioned, but the specific process-mapping techniques used are not detailed. |

**Missing Information and Interview Questions**

**Missing Information and Interview Questions**

*   **SQL Experience:** The job description mentions "advanced spreadsheet experience," but doesn't specify the level of SQL experience required.  Interview questions could include: "Can you describe your experience with SQL?  Are you familiar with any specific SQL functions or techniques?"
*   **Process-Mapping Experience:** The job description emphasizes process mapping.  The interviewer would want to understand the specific types of processes Jordan has mapped.  Interview questions could include: "Can you describe a process mapping project you’ve worked on? What challenges did you face, and how did you overcome them?"
*   **Specifics of Standard Operating Procedures (SOPs):** The interviewer would want to know the type of SOPs Jordan has written.  Are they focused on data entry, quality control, or something else?  Interview questions could include: "Could you give me an example of a SOP you’ve written? What was the goal of that SOP?"
*   **Level of Detail in Procedures:** The interviewer would want to know how detailed the SOPs are.  Are they comprehensive or just high-level guidelines?  Interview questions could include: "Can you walk me through the key steps in a standard operating procedure you've written?"
*   **Data Entry Focus:** The job description highlights the need to reduce duplicate data entry.  The interviewer would want to understand if Jordan has experience with data validation or data cleansing techniques.  Interview questions could include: "Have you ever worked on data validation or cleansing efforts?  What tools or techniques have you used?"
*   **Reporting & Analytics:** The job description mentions presenting findings.  The interviewer would want to understand if Jordan has experience with data visualization or reporting.  Interview questions could include: "Have you ever created reports or dashboards to communicate data insights?"

I hope this detailed breakdown is helpful!

### TODO - REFLECT 🖊

🖊 TODO: Previous outputs reached conclusions such as 78 points or HIRE, but this redesigned output only presented evidence and the status for each requirement and did not make any judgment. It also shows that assumptions of gender pronouns such as “he/his” disappeared and only names or neutral reference were used, demonstrating that the explicit prohibition in the prompt actually worked. And the output endpoint is designed not as a “decision” but as an “interview question,” so that the next step naturally leads to human action.

🖊 TODO: The useful part is the table by requirement, and especially the fact that the SQL item is not created and is honestly marked as "not stated." On the other hand, the part that requires verification is the first requirement, the determination of the length of service. Even though the resume clearly stated three years, the model did not cite this and instead judged it as "partially supported," which is an error that must be corrected by comparing it with the original text.

🖊 TODO: Since the model only organizes the evidence and does not draw conclusions, the judgment of whether the requirements are met, the conduct of interviews, and the final hiring decision remain entirely up to people. In addition, eerrors that remain, such as the years-of-experience error, are still visible in the table format, allowing people to easily verify them against the resume. In the end, the core of this redesign was not to make the model a more deliberate decision-maker, but to change the model’s very role to that of an information organizer, thereby eliminating the possibility from the start that decision-making authority would be transferred to the model.


# Part 3: Business Outcome and Simpler Alternative

Choose one Part 1 project. Start with the result the business needs, not the technology.


### TODO - DECIDE 🖊

Choose **one** project from Part 1.

**Selected project:** 🖊 Case A. Customer retention

**Business outcome:** 🖊 Reduce monthly subscription churn — to identify drop-off customers in advance and carry out retention outreach

**Metric, baseline, and target:** 🖊
Metric: Monthly churn rate

Baseline: Monthly churn rate records over the past 6–12 months (how the current level is, without intervention)

Target: For example, a 1–2 percentage point decrease from the baseline (the specific number will be set after reviewing the baseline)


**Simpler alternative:** 🖊 Simple alternative: rule-based flags.
For example, if two or more of the following apply: 'no login in the last 30 days + pricing plan downgrade + a recent customer service complaint,' the customer is marked as at risk and contacted by the retention team. Can be implemented using existing database queries without an AI model


**Comparison evidence:** 🖊 Comparison evidence: Applying both to historical data for comparison.
The actual churn rate (accuracy) among customers identified by the rule and AI model, the number of missed churn customers, and the deployment and operational costs. Or run both methods side-by-side as pilots and compare the extent of retention improvement

**Initial recommendation:** 🖊 Test simple rules first.
Here are reasons:
(1) They cost almost nothing and can be implemented immediately,
(2) The performance of the rules serves as a baseline against which AI can be measured on how much better it performs,
(3) if the rules alone reaches its goal using only the rules, the AI is unnecessary; if it falls short, that gap becomes the basis for the AI pilot. Judgment is made not by whether the technology is impressive, but by expected value and evidence


# Part 4: Reflect on the Assignment

Integrate what you observed across classification, résumé review, evidence gathering, and process fit.


### TODO - FINAL REFLECTION 🖊

Write **150–200 words** addressing all four questions:

1. What did the model do well and poorly in the classifications?
- The four primary category classifications were generally reasonable, but the secondary category was completely missed, especially the Generative AI-related content in Case D, which was entirely omitted. Moreover, the reasoning presented as its basis was circular, in that the conclusion itself was used again as the basis to justify it.
2. What did the résumé prompts reveal about unsupported precision or authority?
- Even though the same resume and job posting were used, it was only the requested output format changed from a score request to a decision request, yet baseless precision (78 points) and baseless authority ("HIRE") were created. This shows that it is not the data, but the requested format itself that can change the model’s conclusion.
3. How would you use AI to gather evidence without delegating the employment decision?
- The prompt, redefined as “information organizer,” marks each requirement as supported/not stated, explicitly forbids scoring, recommending, or judging, and concludes the output with an interview question, thereby leaving the final judgment entirely to people.
4. Why compare an AI proposal with a simpler process change and measurable outcome?
- The added value of using AI can only be demonstrated when there are measurable outcomes and a baseline for comparison, and if the same results can be achieved using cheaper and simpler methods, then using AI becomes an unnecessary choice.
